# Nowcasting Euro Area GDP with Newspaper Sentiment

Based on Ashwin, Kalamara, and Saiz (2024), *Nowcasting Euro Area GDP with News Sentiment: A Tale of Two Crises*, **Journal of Applied Econometrics**, 39(5), 887–905.

This notebook walks through the core nowcasting exercise from the paper. The goal is to see whether daily newspaper sentiment can improve real-time predictions of quarterly GDP growth for the Euro area, beyond what a standard dynamic factor model (DFM) already provides.

We proceed in three steps:

1. **Explore the data** — understand the structure and visualise the key variables.
2. **Single-day example** — fit the benchmark and text models at one point in time to build intuition.
3. **Full expanding-window evaluation** — run the out-of-sample exercise across 2008–2020 and assess performance.

## 1. Data exploration

The dataset has one row per calendar day from 2004 to 2020. Each row contains:

| Variable | Description |
|---|---|
| `date` | Calendar date |
| `quarter` | Start date of the quarter this day belongs to |
| `moq` | Month of quarter (1, 2, or 3) |
| `doq` | Day of quarter (1 to ~92) |
| `growth_final` | Realised quarterly GDP growth (same for every day in a quarter) |
| `macro_factor_qly` | DFM common factor, updated as macro data arrives through the quarter |
| `macro_factor_qly_1lag` | Previous quarter's factor value |
| `vader_cum`, `econlex_cum`, `stability_cum` | Cumulative within-quarter newspaper sentiment (three different dictionaries) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

df = pd.read_csv("nowcast_data.csv", parse_dates=["date", "quarter"])
df = df.dropna(subset=["macro_factor_qly", "macro_factor_qly_1lag"])

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Quarters: {df['quarter'].nunique()}")
df.head(10)

### Time series of the key variables

To get a quarterly view, we take the **end-of-quarter** value for the factor and sentiment (this is the information set available when the quarter closes) and plot them against realised GDP growth.

In [ ]:
# End-of-quarter snapshots
qly = df.groupby("quarter").agg(
    growth=("growth_final", "first"),
    factor=("macro_factor_qly", "last"),
    vader=("vader_cum", "last"),
).dropna()

# Normalize to zero mean, unit std
growth_z = (qly["growth"] - qly["growth"].mean()) / qly["growth"].std()
factor_z = (qly["factor"] - qly["factor"].mean()) / qly["factor"].std()
vader_z = (qly["vader"] - qly["vader"].mean()) / qly["vader"].std()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# GDP growth and DFM factor (normalized)
ax = axes[0]
ax.plot(qly.index, growth_z, "k-", label="GDP growth")
ax.plot(qly.index, factor_z, "C0--", label="DFM factor")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("Standardized")
ax.legend()
ax.set_title("Euro area GDP growth and DFM factor (normalized)")

# VADER sentiment and GDP growth (normalized)
ax = axes[1]
ax.plot(qly.index, growth_z, "k-", label="GDP growth")
ax.plot(qly.index, vader_z, "C1-", label="VADER sentiment")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("Standardized")
ax.legend()
ax.set_title("GDP growth and cumulative VADER sentiment (normalized)")

plt.tight_layout()
plt.show()

## 2. Model definitions

We nowcast quarterly GDP growth $y_q$ using information available on day $d$ of quarter $q$.

### Benchmark model (DFM only)

$$\hat{y}_{q,d}^{\text{bench}} = \hat{\theta}_0 + \hat{\theta}_1 \, F_{q,d} + \hat{\theta}_2 \, F_{q-1}$$

where $F_{q,d}$ is the DFM factor available on day $d$ of quarter $q$, and $F_{q-1}$ is the previous quarter's factor.

### Text model (DFM + sentiment)

$$\hat{y}_{q,d}^{\text{text}} = \hat{\theta}_0 + \hat{\theta}_1 \, F_{q,d} + \hat{\theta}_2 \, F_{q-1} + \hat{\theta}_3 \, S_{q,d}$$

where $S_{q,d}$ is the cumulative sentiment score up to day $d$ of quarter $q$.

### Training data

To produce a nowcast on day $d$ of quarter $q$, we need training data from past quarters with a comparable information set. The key issue is that predictor values evolve within a quarter: the DFM factor updates whenever new macro releases arrive (and is carried forward between releases), while cumulative sentiment grows daily. If we are on day 15 of the current quarter, using end-of-quarter values from past quarters would give us a misleading training set — those reflect information we don't yet have.

The solution is **day-of-quarter matching**: for each past quarter $q' < q$, we select the single day whose day-of-quarter is closest to $d$. This produces one training observation $(y_{q'}, F_{q',d'}, F_{q'-1}, S_{q',d'})$ per past quarter, where each observation reflects roughly the same position within the quarter as today. Training uses all available past quarters (expanding window), so the sample grows over time.

### Test data

The test observation is a **single point**: the current day's predictor values $(F_{q,d}, F_{q-1}, S_{q,d})$, which we plug into the estimated model to produce a nowcast $\hat{y}_{q,d}$. This is repeated independently for every day $d$ in the evaluation window (2008–2020), each time re-estimating the model on all preceding quarters.

### Evaluation

We compare models using mean squared error over all evaluation days:

$$\text{MSE}_{\text{model}} = \frac{1}{|\mathcal{E}|} \sum_{(q,d) \in \mathcal{E}} \left( y_q - \hat{y}_{q,d} \right)^2$$

and report the percentage gain of the text model over the benchmark:

$$\text{Gain} = 1 - \frac{\text{MSE}_{\text{text}}}{\text{MSE}_{\text{bench}}}$$

A positive gain means sentiment helps.

## 3. Single-day example

Before running the full evaluation, let's pick a single forecast date and walk through the procedure step by step. We'll choose a day in **month 1** of a quarter (when the DFM has the least fresh macro data) to see where sentiment might add the most value.

In [ ]:
# Pick a specific date
forecast_date = pd.Timestamp("2012-04-15")
row = df.loc[df["date"] == forecast_date].iloc[0]

print(f"Forecast date:     {row['date'].date()}")
print(f"Quarter:           {row['quarter'].date()}")
print(f"Month of quarter:  {row['moq']}")
print(f"Day of quarter:    {row['doq']}")
print(f"Realised growth:   {row['growth_final']:.3f}")
print(f"DFM factor:        {row['macro_factor_qly']:.3f}")
print(f"DFM factor (lag):  {row['macro_factor_qly_1lag']:.3f}")
print(f"VADER sentiment:   {row['vader_cum']:.4f}")

### Construct the training set

We match each past quarter to the day closest to our current day-of-quarter position.

In [ ]:
# Training data: one obs per past quarter, matched by doq
sentiment = "vader_cum"
bench_vars = ["macro_factor_qly", "macro_factor_qly_1lag"]
all_vars = [sentiment] + bench_vars
target = "growth_final"

quart = row["quarter"]
today_doq = row["doq"]

# Keep only days from quarters before the current one
past = df[df["quarter"] < quart].copy()
# For each past day, compute how far its day-of-quarter is from today's
past["doq_dist"] = (past["doq"] - today_doq).abs()
# Within each past quarter, pick the single day closest to today's position
train = past.loc[past.groupby("quarter")["doq_dist"].idxmin()]
# Keep only the columns we need and drop any rows with missing predictors
train = train[[target] + all_vars].dropna()

print(f"Training observations: {len(train)} quarters")
print(f"Training period: {past['quarter'].min().date()} to {past['quarter'].max().date()}")
train[[target] + all_vars].describe().round(4)

### Fit both models and compare

In [ ]:
# Benchmark: DFM only
X_bench_train = train[bench_vars].values
y_train = train[target].values

model_bench = LinearRegression().fit(X_bench_train, y_train)
pred_bench = model_bench.predict(row[bench_vars].values.reshape(1, -1))[0]

# Text: DFM + sentiment
X_all_train = train[all_vars].values

model_text = LinearRegression().fit(X_all_train, y_train)
pred_text = model_text.predict(row[all_vars].values.reshape(1, -1))[0]

actual = row[target]

print(f"Actual GDP growth:    {actual:.3f}")
print(f"Benchmark prediction: {pred_bench:.3f}  (error: {actual - pred_bench:+.3f})")
print(f"Text prediction:      {pred_text:.3f}  (error: {actual - pred_text:+.3f})")
print()
print(f"Benchmark coefficients: intercept={model_bench.intercept_:.3f}, "
      f"factor={model_bench.coef_[0]:.3f}, factor_lag={model_bench.coef_[1]:.3f}")
print(f"Text coefficients:      intercept={model_text.intercept_:.3f}, "
      f"factor={model_text.coef_[0]:.3f}, factor_lag={model_text.coef_[1]:.3f}, "
      f"sentiment={model_text.coef_[2]:.3f}")

## 4. Full expanding-window evaluation

Now we run the same procedure for every day from 2008 to 2020, producing a daily nowcast from both models. We test three sentiment dictionaries:

- **VADER** — general-purpose sentiment (Hutto and Gilbert, 2014)
- **ECONLEX** — economics-focused (Barbaglia et al., 2022)
- **Stability** — financial stability dictionary (Correa et al., 2017)

In [ ]:
start_date = pd.Timestamp("2008-01-02")
end_date = pd.Timestamp("2020-12-31")
forecast_idx = df.index[(df["date"] >= start_date) & (df["date"] <= end_date)]

def run_oos(df, sentiment, forecast_idx, make_model=None, min_train=4):
    """
    Out-of-sample expanding-window nowcast evaluation.

    Parameters
    ----------
    df : DataFrame with daily observations
    sentiment : name of the sentiment column
    forecast_idx : index positions to forecast
    make_model : callable returning a sklearn-style estimator
                 (default: LinearRegression)
    min_train : minimum number of training quarters required
    """
    if make_model is None:
        make_model = lambda: LinearRegression()

    all_vars = [sentiment] + bench_vars
    results = []

    for tt in forecast_idx:
        row = df.loc[tt]
        quart = row["quarter"]
        today_doq = row["doq"]

        if row[all_vars].isna().any():
            continue

        past = df[df["quarter"] < quart].copy()
        past["doq_dist"] = (past["doq"] - today_doq).abs()
        train = past.loc[past.groupby("quarter")["doq_dist"].idxmin()]
        train = train[[target] + all_vars].dropna()

        if len(train) < min_train:
            continue

        X_bench = train[bench_vars].values
        X_all = train[all_vars].values
        y = train[target].values

        model_bench = make_model().fit(X_bench, y)
        pred_bench = model_bench.predict(row[bench_vars].values.reshape(1, -1))[0]

        model_all = make_model().fit(X_all, y)
        pred_text = model_all.predict(row[all_vars].values.reshape(1, -1))[0]

        results.append({
            "date": row["date"],
            "quarter": quart,
            "moq": row["moq"],
            "doq": row["doq"],
            "growth_final": row[target],
            "pred_bench": pred_bench,
            "pred_text": pred_text,
            "sentiment": sentiment,
            "n_train": len(train),
        })

    return pd.DataFrame(results)

# --- OLS evaluation across three dictionaries ---
all_preds = []
for sent in ["vader_cum", "econlex_cum", "stability_cum"]:
    print(f"Running OLS: {sent}...")
    p = run_oos(df, sent, forecast_idx)
    all_preds.append(p)

preds = pd.concat(all_preds, ignore_index=True)
print(f"Done. {len(preds)} prediction-days.")

### Overall results

In [ ]:
preds["se_bench"] = (preds["growth_final"] - preds["pred_bench"]) ** 2
preds["se_text"] = (preds["growth_final"] - preds["pred_text"]) ** 2

overall = preds.groupby("sentiment").agg(
    mse_bench=("se_bench", "mean"),
    mse_text=("se_text", "mean"),
).reset_index()
overall["gain_pct"] = ((1 - overall["mse_text"] / overall["mse_bench"]) * 100).round(2)

print("Overall: DFM benchmark vs DFM + sentiment")
print("=" * 55)
print(overall.to_string(index=False))
print("\nPositive gain = sentiment helps.")

### Results by month of quarter

The paper's key finding is that text is most valuable **early in the quarter**, before the DFM has been updated with fresh macro releases.

In [ ]:
by_moq = preds.groupby(["sentiment", "moq"]).agg(
    mse_bench=("se_bench", "mean"),
    mse_text=("se_text", "mean"),
).reset_index()
by_moq["gain_pct"] = ((1 - by_moq["mse_text"] / by_moq["mse_bench"]) * 100).round(2)

print("By month of quarter")
print("=" * 65)
print(by_moq.to_string(index=False))
print("\nMonth 1 = early in quarter, Month 3 = late.")

### MSE by day of quarter

Following the paper's Figure 3, we can plot how the MSE of each model evolves through the quarter.

In [ ]:
# Focus on VADER for the plot
vader_preds = preds[preds["sentiment"] == "vader_cum"].copy()

mse_by_doq = vader_preds.groupby("doq").agg(
    mse_bench=("se_bench", "mean"),
    mse_text=("se_text", "mean"),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax = axes[0]
ax.plot(mse_by_doq["doq"], mse_by_doq["mse_bench"], "C3-", label="DFM benchmark", alpha=0.7)
ax.plot(mse_by_doq["doq"], mse_by_doq["mse_text"], "C2-", label="DFM + VADER", alpha=0.7)
ax.set_ylabel("MSE")
ax.legend()
ax.set_title("Nowcast MSE by day of quarter (VADER)")

ax = axes[1]
gain = (1 - mse_by_doq["mse_text"] / mse_by_doq["mse_bench"]) * 100
ax.bar(mse_by_doq["doq"], gain, color=np.where(gain > 0, "C2", "C3"), alpha=0.6)
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("Day of quarter")
ax.set_ylabel("Gain (%)")
ax.set_title("Percentage MSE reduction from adding VADER sentiment")

plt.tight_layout()
plt.show()

## 5. Non-linear relationship between sentiment and growth

The OLS models above assume a linear relationship between sentiment and GDP growth. But the scatter plot and heatmap below suggest the relationship may be non-linear — sentiment is most informative in the tails, particularly when both the DFM factor and sentiment point in the same (negative) direction.

In [ ]:
# --- Scatter: GDP growth vs VADER sentiment with OLS ---
# Use end-of-quarter snapshots
qly = df.groupby("quarter").agg(
    growth=("growth_final", "first"),
    factor=("macro_factor_qly", "last"),
    vader=("vader_cum", "last"),
).dropna()

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(qly["vader"], qly["growth"], color="grey", alpha=0.5, edgecolors="k", linewidths=0.5)

# OLS fit
from numpy.polynomial.polynomial import polyfit
b, m = polyfit(qly["vader"], qly["growth"], 1)
x_line = np.linspace(qly["vader"].min(), qly["vader"].max(), 100)
ax.plot(x_line, b + m * x_line, "b--", label="OLS")

ax.set_xlabel("Cumulative VADER Sentiment (end of quarter)")
ax.set_ylabel("GDP Growth (%)")
ax.set_title("GDP Growth vs VADER Sentiment")
ax.legend()
plt.tight_layout()
plt.show()

# --- Heatmap: average GDP growth by DFM x Sentiment terciles ---
qly = qly.copy()
qly["vader_tercile"] = pd.qcut(qly["vader"], 3, labels=["Low", "Mid", "High"])
qly["factor_tercile"] = pd.qcut(qly["factor"], 3, labels=["Low", "Mid", "High"])

heatmap_data = qly.groupby(["factor_tercile", "vader_tercile"]).agg(
    mean_growth=("growth", "mean"),
    n=("growth", "count"),
).reset_index()

pivot_growth = heatmap_data.pivot(index="factor_tercile", columns="vader_tercile", values="mean_growth")
pivot_n = heatmap_data.pivot(index="factor_tercile", columns="vader_tercile", values="n")

# Reorder so High is on top
pivot_growth = pivot_growth.loc[["High", "Mid", "Low"], ["Low", "Mid", "High"]]
pivot_n = pivot_n.loc[["High", "Mid", "Low"], ["Low", "Mid", "High"]]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot_growth.values, cmap="RdBu", aspect="auto",
               vmin=-2, vmax=2.5)

# Annotate cells — use black text for readability
for i in range(3):
    for j in range(3):
        val = pivot_growth.values[i, j]
        n = int(pivot_n.values[i, j])
        ax.text(j, i, f"{val:.2g}\n(n={n})", ha="center", va="center",
                fontsize=13, fontweight="bold", color="black")

ax.set_xticks(range(3))
ax.set_xticklabels(["Low", "Mid", "High"])
ax.set_yticks(range(3))
ax.set_yticklabels(["High", "Mid", "Low"])
ax.set_xlabel("VADER Sentiment (tercile)")
ax.set_ylabel("DFM Factor (tercile)")
ax.set_title("Average GDP Growth by DFM x Sentiment")
fig.colorbar(im, ax=ax, label="Mean GDP Growth (%)", shrink=0.8)
plt.tight_layout()
plt.show()

## 6. Random Forest

### From trees to forests

A **regression tree** partitions the covariate space $\mathcal{X}$ into $M$ non-overlapping regions and assigns a constant prediction in each:

$$f_\theta(\mathbf{x}) = \sum_{m=1}^{M} \gamma_m \, \mathbf{1}(\mathbf{x} \in R_m)$$

where $\gamma_m = \bar{y}_m$, the mean outcome for training observations in region $R_m$. The regions $\{R_m\}$ are found by a greedy algorithm that recursively splits on individual covariates to minimise RSS:

$$\sum_{i:\, \mathbf{x}_i \in R_1} (y_i - \bar{y}_1)^2 + \sum_{i:\, \mathbf{x}_i \in R_2} (y_i - \bar{y}_2)^2$$

Individual trees are easy to interpret but have **high variance**: a different draw of training data is likely to produce different splits. This makes their out-of-sample performance poor.

**Bagging** (bootstrap aggregation) reduces this variance by:
1. Drawing $S$ bootstrap samples from the training data.
2. Fitting an unpruned tree $\hat{f}_s$ on each sample.
3. Averaging: $\hat{f}(\mathbf{x}) = \frac{1}{S} \sum_s \hat{f}_s(\mathbf{x})$.

A **random forest** extends bagging by restricting each split to a random subset of $\tilde{J}$ out of the $J$ available covariates (typically $\tilde{J} = \lfloor\sqrt{J}\rfloor$). This **de-correlates** the individual trees: if one predictor is very strong, not every tree will split on it first. Averaging less correlated predictions is more effective at reducing error.

### Why try this here?

The scatter plot and heatmap above suggest the relationship between sentiment and growth may be non-linear — sentiment is most informative in the tails. A random forest can capture this without us having to specify the functional form. The paper finds that nonlinear models help during extreme episodes (the Great Lockdown) but require sufficient training data to be effective.

### Single-day example with Random Forest

We return to the same forecast date as in Section 3 and fit a random forest alongside OLS. The key `scikit-learn` hyperparameters are:

| Parameter | Value | Role |
|---|---|---|
| `n_estimators` | 100 | Number of bootstrap trees $S$ |
| `max_features` | `None` (= all features) | Covariates considered per split $\tilde{J}$ — with only 2–3 predictors, restricting further would starve each split |
| `min_samples_leaf` | 1 (default) | Minimum observations in a terminal node — the default allows trees to capture nonlinear tail behaviour, which is where sentiment is most informative |
| `random_state` | 42 | Seed for reproducibility |

With only ~28 training observations, one might consider increasing `min_samples_leaf` to regularise. However, the performance gains from RF come precisely from capturing nonlinear tail behaviour (sentiment is most informative during crises), and over-regularising squashes this. We keep the default of 1.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Same forecast date as Section 3
forecast_date = pd.Timestamp("2012-04-15")
row = df.loc[df["date"] == forecast_date].iloc[0]

sentiment = "vader_cum"
bench_vars = ["macro_factor_qly", "macro_factor_qly_1lag"]
text_vars = [sentiment] + bench_vars
target = "growth_final"

# Reconstruct training set (same as Section 3)
quart = row["quarter"]
today_doq = row["doq"]
past = df[df["quarter"] < quart].copy()
past["doq_dist"] = (past["doq"] - today_doq).abs()
train = past.loc[past.groupby("quarter")["doq_dist"].idxmin()]
train = train[[target] + text_vars].dropna()

X_text_train = train[text_vars].values
X_bench_train = train[bench_vars].values
y_train = train[target].values

print(f"Forecast date: {forecast_date.date()}")
print(f"Training quarters: {len(train)}")
print(f"Features (text model): {text_vars}")
print()

# --- OLS (for comparison) ---
ols_bench = LinearRegression().fit(X_bench_train, y_train)
ols_text = LinearRegression().fit(X_text_train, y_train)

pred_ols_bench = ols_bench.predict(row[bench_vars].values.reshape(1, -1))[0]
pred_ols_text = ols_text.predict(row[text_vars].values.reshape(1, -1))[0]

# --- Random Forest ---
rf_params = dict(
    n_estimators=100,
    max_features=None,       # use all features at each split (only 2-3 vars)
    random_state=42,
)

rf_bench = RandomForestRegressor(**rf_params).fit(X_bench_train, y_train)
rf_text = RandomForestRegressor(**rf_params).fit(X_text_train, y_train)

pred_rf_bench = rf_bench.predict(row[bench_vars].values.reshape(1, -1))[0]
pred_rf_text = rf_text.predict(row[text_vars].values.reshape(1, -1))[0]

actual = row[target]

print(f"Actual GDP growth:  {actual:.3f}")
print()
print(f"{'Model':<25s} {'Prediction':>10s} {'Error':>10s}")
print("-" * 47)
print(f"{'OLS: DFM only':<25s} {pred_ols_bench:10.3f} {actual - pred_ols_bench:+10.3f}")
print(f"{'OLS: DFM + VADER':<25s} {pred_ols_text:10.3f} {actual - pred_ols_text:+10.3f}")
print(f"{'RF:  DFM only':<25s} {pred_rf_bench:10.3f} {actual - pred_rf_bench:+10.3f}")
print(f"{'RF:  DFM + VADER':<25s} {pred_rf_text:10.3f} {actual - pred_rf_text:+10.3f}")

We can also inspect how the random forest uses the covariates. With `feature_importances_`, `scikit-learn` reports the average reduction in RSS attributable to each feature across all splits and all trees.

In [ ]:
print("RF feature importances (DFM + VADER model):")
for name, imp in zip(text_vars, rf_text.feature_importances_):
    print(f"  {name:<30s} {imp:.3f}")

### Full expanding-window evaluation with Random Forest

We now run the same daily expanding-window exercise as in Section 4, but using Random Forest instead of OLS. We focus on VADER since it performed best in the OLS exercise.

One important difference: we require at least **10 training quarters** (vs 4 for OLS) since the random forest needs more data to estimate meaningful splits.

In [ ]:
make_rf = lambda: RandomForestRegressor(
    n_estimators=100,
    max_features=None,
    random_state=42,
)

print("Running Random Forest: vader_cum...")
rf_preds = run_oos(df, "vader_cum", forecast_idx, make_model=make_rf, min_train=10)
print(f"Done. {len(rf_preds)} prediction-days.")

### RF results: overall and by month of quarter

In [ ]:
rf_preds["se_bench"] = (rf_preds["growth_final"] - rf_preds["pred_bench"]) ** 2
rf_preds["se_text"] = (rf_preds["growth_final"] - rf_preds["pred_text"]) ** 2

# Overall
mse_b = rf_preds["se_bench"].mean()
mse_t = rf_preds["se_text"].mean()
gain = (1 - mse_t / mse_b) * 100

print("Random Forest: DFM benchmark vs DFM + VADER")
print("=" * 50)
print(f"MSE (DFM only):    {mse_b:.4f}")
print(f"MSE (DFM + VADER): {mse_t:.4f}")
print(f"Gain:              {gain:.2f}%")

# By month of quarter
print()
rf_moq = rf_preds.groupby("moq").agg(
    mse_bench=("se_bench", "mean"),
    mse_text=("se_text", "mean"),
).reset_index()
rf_moq["gain_pct"] = ((1 - rf_moq["mse_text"] / rf_moq["mse_bench"]) * 100).round(2)
print("By month of quarter:")
print(rf_moq.to_string(index=False))

### Comparing OLS and RF across the quarter

We can now compare all four models — OLS and RF, each with and without sentiment — to see where nonlinearity helps.

In [ ]:
# Merge OLS VADER preds with RF preds
vader_ols = preds[preds["sentiment"] == "vader_cum"][["doq", "se_bench", "se_text"]].copy()
vader_ols = vader_ols.rename(columns={"se_bench": "se_ols_bench", "se_text": "se_ols_text"})

ols_by_doq = vader_ols.groupby("doq").agg(
    mse_ols_bench=("se_ols_bench", "mean"),
    mse_ols_text=("se_ols_text", "mean"),
).reset_index()

rf_by_doq = rf_preds.groupby("doq").agg(
    mse_rf_bench=("se_bench", "mean"),
    mse_rf_text=("se_text", "mean"),
).reset_index()

merged = ols_by_doq.merge(rf_by_doq, on="doq")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# MSE levels
ax = axes[0]
ax.plot(merged["doq"], merged["mse_ols_bench"], "C3-", alpha=0.6, label="OLS: DFM only")
ax.plot(merged["doq"], merged["mse_ols_text"], "C2-", alpha=0.6, label="OLS: DFM + VADER")
ax.plot(merged["doq"], merged["mse_rf_bench"], "C3--", alpha=0.6, label="RF: DFM only")
ax.plot(merged["doq"], merged["mse_rf_text"], "C2--", alpha=0.6, label="RF: DFM + VADER")
ax.set_ylabel("MSE")
ax.legend(fontsize=9)
ax.set_title("Nowcast MSE by day of quarter: OLS vs Random Forest")

# Gain of adding VADER
ax = axes[1]
gain_ols = (1 - merged["mse_ols_text"] / merged["mse_ols_bench"]) * 100
gain_rf = (1 - merged["mse_rf_text"] / merged["mse_rf_bench"]) * 100
ax.plot(merged["doq"], gain_ols, "C0-", label="OLS gain from VADER")
ax.plot(merged["doq"], gain_rf, "C1--", label="RF gain from VADER")
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("Day of quarter")
ax.set_ylabel("Gain (%)")
ax.legend(fontsize=9)
ax.set_title("MSE reduction from adding VADER sentiment")

plt.tight_layout()
plt.show()

### Effect of crisis times

In [ ]:
# Split into crisis and normal periods
crisis_quarters = pd.to_datetime([
    "2008-07-01", "2008-10-01", "2009-01-01", "2009-04-01",  # Great Recession
    "2020-01-01", "2020-04-01", "2020-07-01", "2020-10-01",  # Great Lockdown
])

# OLS VADER predictions
vader_ols = preds[preds["sentiment"] == "vader_cum"].copy()
vader_ols["crisis"] = vader_ols["quarter"].isin(crisis_quarters)

# RF predictions
rf_preds["crisis"] = rf_preds["quarter"].isin(crisis_quarters)

for label, is_crisis in [("Crisis quarters", True), ("Normal quarters", False)]:
    print(f"\n{'=' * 55}")
    print(f"  {label}")
    print(f"{'=' * 55}")
    
    ols_sub = vader_ols[vader_ols["crisis"] == is_crisis]
    rf_sub = rf_preds[rf_preds["crisis"] == is_crisis]
    
    ols_mse_b = ols_sub["se_bench"].mean()
    ols_mse_t = ols_sub["se_text"].mean()
    rf_mse_b = rf_sub["se_bench"].mean()
    rf_mse_t = rf_sub["se_text"].mean()
    
    print(f"{'Model':<25s} {'MSE bench':>10s} {'MSE text':>10s} {'Gain':>8s}")
    print("-" * 55)
    print(f"{'OLS':<25s} {ols_mse_b:10.4f} {ols_mse_t:10.4f} {(1-ols_mse_t/ols_mse_b)*100:7.2f}%")
    print(f"{'Random Forest':<25s} {rf_mse_b:10.4f} {rf_mse_t:10.4f} {(1-rf_mse_t/rf_mse_b)*100:7.2f}%")
    print(f"\n  n days: OLS={len(ols_sub)}, RF={len(rf_sub)}")